# Aircraft Maintenance Manuals: Ingestion & Vector Search

Builds the searchable manual library that backs the **Maintenance Manuals** tab of `aircraft-maintenance-app`.

1. Generate one synthetic maintenance manual (PDF) per fleet aircraft type. Each manual has a troubleshooting task for every fault type seen in the fleet (Brake Overheat, Vibration Anomaly, ...).
2. Land the PDFs in the UC volume `genie_zeroops_mfg_catalog.default.maintenance_manuals`.
3. Parse them with `ai_parse_document` and save to `maintenance_manual_parsed`.
4. Split them into one chunk per task, with part number, ATA chapter, system and fault code, and save to `maintenance_manual_chunks`.
5. Build a Vector Search delta-sync index (`maintenance_manual_index`) on a small endpoint.
6. Grant the app's service principal read access.

> The manuals are **synthetic demo content**, not OEM documentation. Never use them for real maintenance.

In [0]:
%pip install reportlab --quiet
dbutils.library.restartPython()

In [0]:
CATALOG, SCHEMA = "genie_zeroops_mfg_catalog", "default"
VOLUME = "maintenance_manuals"
VOL_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"
RAW_DIR = f"{VOL_PATH}/raw"
PARSED_TABLE = f"{CATALOG}.{SCHEMA}.maintenance_manual_parsed"
CHUNK_TABLE = f"{CATALOG}.{SCHEMA}.maintenance_manual_chunks"
VS_ENDPOINT = "aircraft-manuals-vs"
VS_INDEX = f"{CATALOG}.{SCHEMA}.maintenance_manual_index"
EMBED_MODEL = "databricks-gte-large-en"
APP_SP = "a048a692-d417-4d9f-a196-84205b8314b3"   # aircraft-maintenance-app service principal

spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.{VOLUME} COMMENT 'Synthetic aircraft maintenance manuals (PDF) for the maintenance search demo'")
dbutils.fs.mkdirs(RAW_DIR)
print("Volume ready:", RAW_DIR)

In [0]:
import os, shutil
from reportlab.lib.pagesizes import letter
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import inch
from reportlab.platypus import SimpleDocTemplate, Paragraph as _RLParagraph, Spacer, PageBreak
from xml.sax.saxutils import escape as _esc

def Paragraph(text, style):
    """reportlab treats text as XML markup; escape '&', '<', '>' in manual content."""
    return _RLParagraph(_esc(text), style)

# Fleet aircraft types (from aircraft_registry) -> OEM, engine, short code used in part numbers
aircraft_types = [r.aircraft_type for r in spark.sql(
    "SELECT DISTINCT aircraft_type FROM genie_zeroops_mfg_catalog.aircraft_maintenance.aircraft_registry ORDER BY 1").collect()]
AIRCRAFT = {
    'A320neo':    ('Airbus', 'CFM LEAP-1A', 'A20N'),
    'A321neo':    ('Airbus', 'Pratt & Whitney PW1100G-JM', 'A21N'),
    'A330-900':   ('Airbus', 'Rolls-Royce Trent 7000', 'A339'),
    'A350-900':   ('Airbus', 'Rolls-Royce Trent XWB-84', 'A359'),
    'B737-800':   ('Boeing', 'CFM56-7B', 'B738'),
    'B767-300':   ('Boeing', 'GE CF6-80C2', 'B763'),
    'B777-300ER': ('Boeing', 'GE90-115B', 'B77W'),
    'B787-9':     ('Boeing', 'GEnx-1B', 'B789'),
    'CRJ-900':    ('Bombardier', 'GE CF34-8C5', 'CRJ9'),
    'E190-E2':    ('Embraer', 'Pratt & Whitney PW1900G', 'E295'),
}
missing = [t for t in aircraft_types if t not in AIRCRAFT]
assert not missing, f"Add OEM/engine mapping for: {missing}"

# One troubleshooting task per fleet fault code (matches anomaly_type in the gold tables)
FAULTS = [
  dict(code='BRAKE_OVERHEAT', ata='32-42-01', system='Landing Gear - Wheels & Brakes', title='Wheel Brake Overheat',
       part='Carbon Brake Heat Stack Assembly', pn='3242-117', extra_parts=['Brake Temperature Sensor (pn {c}-3246-022)', 'Wheel Fuse Plug Set (pn {c}-3241-008)'],
       symptoms='BRAKES HOT advisory on landing rollout; brake temperature above 300 C on one or more wheels; fuse plugs melted; smell of burning or smoke at the main gear.',
       causes='Dragging brake from a sticking piston or worn return springs; excessive braking during short-runway or high-energy landing; contaminated or glazed carbon heat stack; failed brake temperature sensor giving false indication.',
       limits='Heat stack wear pin protrusion must exceed 0.25 in (6.4 mm). Brake temperature must cool below 150 C before inspection. Replace the wheel if any fuse plug has melted.',
       steps=['Chock the aircraft and allow the brakes to cool below 150 C. Do not approach a hot wheel from the side; approach fore or aft only.',
              'Read the brake temperature monitoring page and compare all wheels; a single hot wheel indicates a dragging brake.',
              'Inspect wear pins on the affected brake. If protrusion is below 0.25 in, replace the heat stack assembly.',
              'Inspect fuse plugs. If any plug has melted, remove the wheel and tire assembly and send it to the wheel shop.',
              'Release parking brake and rotate the wheel by hand; resistance indicates a sticking piston. Replace the brake assembly if drag persists.',
              'Perform a brake bleed and a functional test of the anti-skid system per the test task.',
              'Replace the brake temperature sensor if its reading differs from a calibrated probe by more than 20 C.'],
       torque='Brake assembly retaining nuts: 450-500 in-lb. Axle nut: torque per wheel installation task, then back off and re-torque.'),
  dict(code='VIBRATION_ANOMALY', ata='72-31-05', system='Engine - Fan & Compressor', title='Engine High Vibration (N1/N2)',
       part='Fan Blade Set and Fan Trim Balance Weights', pn='7231-204', extra_parts=['No. 1 Bearing Vibration Accelerometer (pn {c}-7731-015)', 'Fan Blade Dovetail Shim Kit (pn {c}-7231-210)'],
       symptoms='ENG VIB amber indication; N1 vibration above 4.0 units or N2 vibration above 3.0 units; crew report of airframe buzz in climb.',
       causes='Fan imbalance from blade erosion, ice or FOD damage; loose or missing trim balance weights; worn dovetail shims; failed accelerometer or loose sensor connector; No. 1 bearing degradation.',
       limits='N1 vibration limit: 4.0 units continuous. Fan blade leading-edge damage limits per blade-damage chart; any crack is cause for replacement.',
       steps=['Download the engine vibration monitoring data and identify the rotor (N1 or N2) and flight phase of the peak.',
              'Perform a visual borescope-free inspection of the fan blades for FOD, erosion and ice damage.',
              'Check fan blade dovetails and shims for wear and migration; replace the shim kit if worn.',
              'Inspect the accelerometer and its harness connector; secure or replace if loose.',
              'Perform a fan trim balance run using the onboard balance solution; install trim weights as calculated.',
              'If N2 vibration remains above limits, perform a core borescope inspection and analyze oil debris for bearing distress.'],
       torque='Fan trim balance weight screws: 55-65 in-lb, safety-wire after installation.'),
  dict(code='BLEED_AIR_LEAK', ata='36-11-02', system='Pneumatic - Engine Bleed Air', title='Bleed Air Leak / Duct Overheat',
       part='High Pressure Bleed Valve (HPV)', pn='3611-330', extra_parts=['Bleed Duct V-Band Clamp (pn {c}-3612-041)', 'Bleed Leak Detection Loop (pn {c}-3615-007)'],
       symptoms='BLEED LEAK or WING/BODY OVHT warning; bleed pressure low on one side; pack temperature fluctuation; discoloration on duct insulation.',
       causes='Cracked bleed duct or failed V-band clamp seal; HP bleed valve stuck open or leaking past the seat; failed precooler; chafed leak-detection loop giving false warnings.',
       limits='No bleed air leakage permitted at duct joints. Duct insulation discoloration larger than 2 in (50 mm) requires duct inspection.',
       steps=['Isolate the affected bleed system and confirm the warning against the maintenance computer fault history.',
              'Open the pylon and wing-to-body fairing access panels and inspect bleed ducts for cracks, discoloration and loose clamps.',
              'Perform a duct leak check with a pressurized dry-air source and leak-detection fluid at each joint.',
              'Replace V-band clamps and seals at any leaking joint.',
              'Perform an operational test of the HP bleed valve; replace the valve if it fails to close within limits.',
              'Test the leak-detection loop continuity and resistance; replace any chafed section.'],
       torque='V-band clamp T-bolt nut: 60-70 in-lb, tap the clamp around its circumference and re-torque.'),
  dict(code='HYDRAULIC_LEAK', ata='29-11-04', system='Hydraulic Power', title='Hydraulic Fluid Leak / Low Quantity',
       part='Engine-Driven Hydraulic Pump (EDP)', pn='2911-450', extra_parts=['Hydraulic Pressure Line Seal Kit (pn {c}-2912-118)', 'Case Drain Filter (pn {c}-2915-033)'],
       symptoms='HYD QTY LOW or HYD PRESS LOW caution; reservoir quantity dropping between flights; fluid on the ramp under the wing or gear bay.',
       causes='Failed EDP shaft seal; loose B-nut or damaged O-ring on a pressure line; chafed hose; actuator rod seal leakage.',
       limits='EDP shaft seal drain: maximum 20 drops per minute. Static leaks at fittings: none permitted. Case drain flow per pump test limits.',
       steps=['Depressurize the hydraulic system and relieve reservoir pressure before opening any line.',
              'Clean the suspect area and run the system on ground power to locate the leak source.',
              'Measure EDP shaft seal drain rate; replace the pump if above 20 drops per minute.',
              'Replace O-rings and re-torque B-nuts on any leaking fitting; replace chafed hoses.',
              'Replace the case drain filter and inspect it for metal contamination.',
              'Service the reservoir to the full mark and perform a leak check at operating pressure.'],
       torque='Pressure line B-nuts: per fitting size table (example -8 size: 270-300 in-lb).'),
  dict(code='COMPRESSOR_STALL', ata='72-35-01', system='Engine - High Pressure Compressor', title='Compressor Stall / Surge',
       part='Variable Bleed Valve (VBV) Actuator', pn='7235-512', extra_parts=['Variable Stator Vane (VSV) Feedback Sensor (pn {c}-7532-019)', 'HPC Stage 1 Blade Set (pn {c}-7235-601)'],
       symptoms='Loud bang with EGT spike and N2 fluctuation; ENG STALL message; flames from the exhaust or inlet reported by crew; usually during climb or high thrust.',
       causes='VBV or VSV actuator rigging error or slow response; HPC blade damage from FOD or erosion; inlet distortion from crosswind; contaminated compressor.',
       limits='Any confirmed stall requires inspection before next flight. HPC blade damage limits per borescope inspection chart.',
       steps=['Retrieve the engine fault history and EGT/N2 trend for the stall event.',
              'Perform a VBV and VSV rigging check and actuator operational test; replace the actuator if response time exceeds limits.',
              'Check the VSV feedback sensor signal; replace if erratic.',
              'Perform an HPC borescope inspection for blade tip curl, nicks and erosion.',
              'Perform an engine water wash if the compressor is contaminated.',
              'Perform a ground run with an acceleration and deceleration check.'],
       torque='VBV actuator rod-end bolts: 95-110 in-lb.'),
  dict(code='ENGINE_DEGRADATION', ata='72-00-10', system='Engine - Performance Monitoring', title='Engine Performance Degradation (EGT Margin Loss)',
       part='HPT Stage 1 Blade Set', pn='7250-720', extra_parts=['HPT Shroud Segment Set (pn {c}-7251-044)', 'Engine Wash Kit (pn {c}-7200-905)'],
       symptoms='EGT margin trending down more than 15 C over 30 days; higher fuel flow for the same thrust; EGT exceedance at takeoff in hot conditions.',
       causes='Compressor fouling; HPT blade erosion or coating loss; increased tip clearance from worn shrouds; seal leakage.',
       limits='Minimum EGT margin for continued operation: 10 C. Plan engine removal when the margin is predicted to reach 0 C within 300 cycles.',
       steps=['Review engine trend monitoring for EGT margin, fuel flow and N2 deltas.',
              'Perform an on-wing compressor water wash and re-baseline the trend.',
              'If no recovery, perform an HPT borescope inspection for blade erosion, burning and coating loss.',
              'Inspect HPT shrouds for wear and rub.',
              'Raise an engine removal plan if borescope findings exceed on-wing limits.'],
       torque='Borescope port plugs: 80-90 in-lb, safety-wire.'),
  dict(code='OIL_SYSTEM_FAULT', ata='79-21-03', system='Engine Oil System', title='Engine Oil Pressure Low / Oil Temperature High',
       part='Engine Oil Pressure Pump & Lube Unit', pn='7921-310', extra_parts=['Main Oil Filter Element (pn {c}-7922-014)', 'Magnetic Chip Detector (pn {c}-7935-002)'],
       symptoms='OIL PRESS LOW caution or oil pressure fluctuating; oil temperature above 140 C; high oil consumption; chip detector warning.',
       causes='Clogged main oil filter; failed lube pump; worn bearing seals causing consumption; failed pressure transmitter; low oil quantity.',
       limits='Oil consumption limit: 0.4 qt per hour. Any metal on the chip detector larger than 1/16 in requires a lab analysis before further flight.',
       steps=['Check oil quantity and the servicing record; calculate consumption per flight hour.',
              'Remove and inspect the main oil filter element; check the bypass indicator.',
              'Remove and inspect the magnetic chip detector; send debris to the lab.',
              'Check the oil pressure transmitter against a calibrated gauge.',
              'Replace the lube and scavenge pump if pressure remains low with a clean filter and serviceable transmitter.',
              'Perform a ground run and leak check.'],
       torque='Chip detector: 45-55 in-lb, safety-wire. Filter cover bolts: 85-95 in-lb.'),
  dict(code='ELECTRICAL_FAULT', ata='24-21-02', system='Electrical Power - AC Generation', title='Generator / AC Bus Fault',
       part='Integrated Drive Generator (IDG)', pn='2421-600', extra_parts=['Generator Control Unit (GCU) (pn {c}-2422-010)', 'Bus Power Control Unit (pn {c}-2450-031)'],
       symptoms='GEN OFF or IDG DISC caution; AC bus transfer; IDG oil temperature high; flickering cabin lights.',
       causes='IDG oil overheat or low oil level; failed GCU; loose feeder cable terminal; failed current transformer.',
       limits='IDG oil temperature rise must not exceed 20 C above ambient at idle. Feeder terminal resistance per wiring practices limits.',
       steps=['Retrieve GCU fault codes from the maintenance computer.',
              'Check IDG oil level and filter differential pressure indicator; service or replace the filter as required.',
              'Inspect feeder cable terminals for heat discoloration and re-torque.',
              'Swap the GCU with the opposite side to isolate the fault.',
              'Replace the IDG if the fault follows the IDG and oil overheat persists. Never reconnect a disconnected IDG in flight.',
              'Perform a generator operational test on engine run.'],
       torque='IDG feeder terminal nuts: 95-105 in-lb.'),
  dict(code='FUEL_IMBALANCE', ata='28-41-01', system='Fuel - Quantity Indication', title='Fuel Imbalance / Quantity Indication Fault',
       part='Fuel Quantity Probe (Capacitance Type)', pn='2841-220', extra_parts=['Fuel Quantity Processor Unit (pn {c}-2842-005)', 'Crossfeed Valve Actuator (pn {c}-2822-017)'],
       symptoms='FUEL IMBAL caution; wing tanks differ by more than 500 kg; unexpected quantity change during cruise; crossfeed required to correct.',
       causes='Contaminated or failed capacitance probe; failed fuel quantity processor channel; crossfeed valve leaking; fuel leak on one wing.',
       limits='Maximum lateral imbalance per weight-and-balance limits. Probe insulation resistance per test limits.',
       steps=['Compare indicated quantity against a dripstick or magnetic level indicator reading.',
              'Run the fuel quantity system BITE test and record failed probes.',
              'Check the crossfeed valve position and actuator; replace if it does not seat closed.',
              'Defuel the tank as required and replace failed probes; clean contaminated probes.',
              'Inspect the wing lower surface for fuel leaks before return to service.'],
       torque='Probe mounting screws: 20-25 in-lb.'),
  dict(code='SENSOR_DRIFT', ata='31-32-04', system='Indicating & Recording - Sensors', title='Sensor Drift / Erroneous Indication',
       part='EGT Thermocouple Harness', pn='7732-140', extra_parts=['N1 Speed Sensor (pn {c}-7711-006)', 'Data Acquisition Unit (pn {c}-3132-001)'],
       symptoms='Parameter reading diverges from the opposite engine or the model; intermittent spikes; data acquisition unit fault messages; no corresponding physical symptom.',
       causes='Thermocouple probe degradation; corroded or chafed harness; loose connector; data acquisition unit channel fault.',
       limits='Thermocouple loop resistance per harness test limits. Sensor reading tolerance: within 2 percent of calibrated reference.',
       steps=['Compare the suspect parameter against the opposite engine and redundant sources.',
              'Perform a thermocouple harness resistance and insulation test.',
              'Inspect connectors for corrosion and bent pins; clean or replace.',
              'Replace the failed probe or harness segment.',
              'Re-run the data acquisition unit BITE and confirm the reading on a ground run.'],
       torque='Thermocouple probe nuts: 30-35 in-lb.'),
]

styles = getSampleStyleSheet()
H1 = ParagraphStyle('H1', parent=styles['Title'], fontSize=22, spaceAfter=14)
H2 = ParagraphStyle('H2', parent=styles['Heading2'], fontSize=14, spaceBefore=10, spaceAfter=6)
B = ParagraphStyle('B', parent=styles['BodyText'], fontSize=10, leading=13)

def build_manual(ac):
    oem, engine, c = AIRCRAFT[ac]
    fname = f"{ac.replace(' ', '_')}_Maintenance_Manual.pdf"
    path = os.path.join('/tmp', fname)
    doc = SimpleDocTemplate(path, pagesize=letter, leftMargin=0.8*inch, rightMargin=0.8*inch, topMargin=0.8*inch, bottomMargin=0.8*inch,
                            title=f"{ac} Aircraft Maintenance Manual", author='Synthetic demo content')
    s = [Paragraph(f"{oem} {ac} Aircraft Maintenance Manual", H1),
         Paragraph(f"Troubleshooting and Repair Supplement - Engines: {engine}", styles['Heading3']),
         Paragraph(f"Aircraft Type: {ac}. Manufacturer: {oem}. Engine: {engine}. Revision 14, October 2026.", B),
         Paragraph("SYNTHETIC DEMO CONTENT. This manual was generated for a Databricks demonstration and is not approved OEM data. Do not use for real maintenance.", B),
         Spacer(1, 10),
         Paragraph("General Safety", H2),
         Paragraph("Obey all WARNINGS and CAUTIONS. Make sure the aircraft is grounded, chocked and that applicable systems are de-energized and tagged before maintenance. Use approved personal protective equipment for hydraulic fluid, fuel and hot surfaces.", B),
         PageBreak()]
    for f in FAULTS:
        s += [Paragraph(f"TASK {f['ata']} {f['title']} - {ac}", H2),
              Paragraph(f"Fault Code: {f['code']}", B),
              Paragraph(f"Aircraft Type: {ac}", B),
              Paragraph(f"ATA Chapter: {f['ata']}", B),
              Paragraph(f"System: {f['system']}", B),
              Paragraph(f"Part Name: {f['part']}", B),
              Paragraph(f"Part Number: {c}-{f['pn']}", B),
              Paragraph("Related Parts: " + '; '.join(p.format(c=c) for p in f['extra_parts']), B),
              Paragraph(f"Symptoms: {f['symptoms']}", B),
              Paragraph(f"Probable Causes: {f['causes']} On the {engine} installation, refer also to the engine manual for engine-specific limits.", B),
              Paragraph(f"Limits: {f['limits']}", B),
              Paragraph("Corrective Action:", B)]
        s += [Paragraph(f"Step {i}. {st}", B) for i, st in enumerate(f['steps'], 1)]
        s += [Paragraph(f"Torque Values: {f['torque']}", B),
              Paragraph(f"Return to Service: Complete the operational test, clear the fault history, and record the action against fault code {f['code']} in the technical log.", B),
              Spacer(1, 8)]
    doc.build(s)
    shutil.copy(path, f"{RAW_DIR}/{fname}")
    return fname

files = [build_manual(ac) for ac in aircraft_types]
print(f"Generated {len(files)} manuals x {len(FAULTS)} tasks:")
for f in dbutils.fs.ls(RAW_DIR):
    print(f"  {f.name}  {f.size/1024:.0f} KB")

In [0]:
%sql
CREATE OR REPLACE TABLE genie_zeroops_mfg_catalog.default.maintenance_manual_parsed
COMMENT 'Synthetic aircraft maintenance manuals parsed with ai_parse_document (v2). One row per PDF.'
AS
WITH parsed AS (
  SELECT _metadata.file_path AS file_path,
         _metadata.file_name AS file_name,
         ai_parse_document(content, MAP('version', '2.0', 'descriptionElementTypes', '')) AS parsed
  FROM READ_FILES('/Volumes/genie_zeroops_mfg_catalog/default/maintenance_manuals/raw/', format => 'binaryFile')
)
SELECT file_path, file_name, parsed,
       concat_ws('\n\n', transform(try_cast(parsed:document:elements AS ARRAY<VARIANT>),
                                    e -> try_cast(e:content AS STRING))) AS full_text,
       is_variant_null(parsed:error_status) AS parse_ok,
       current_timestamp() AS parsed_at
FROM parsed;

SELECT file_name, parse_ok, length(full_text) AS chars FROM genie_zeroops_mfg_catalog.default.maintenance_manual_parsed ORDER BY file_name;

In [0]:
import re
from pyspark.sql import Row

CHUNK_TABLE = "genie_zeroops_mfg_catalog.default.maintenance_manual_chunks"
LABELS = ["Fault Code", "Aircraft Type", "ATA Chapter", "System", "Part Name", "Part Number", "Related Parts",
          "Symptoms", "Probable Causes", "Limits", "Corrective Action", "Torque Values", "Return to Service"]
_NEXT = r"(?=\n\s*(?:" + "|".join(LABELS) + r"):|\n\s*Step \d+\.|\Z)"
_clean = lambda s: re.sub(r"\s+", " ", s).strip() if s else None

def FIELD(label, txt):
    """Value after 'Label:' up to the next known label (parsed text wraps long lines)."""
    m = re.search(rf"{label}:\s*(.+?){_NEXT}", txt, re.S)
    return _clean(m.group(1)) if m else None

rows = []
for d in spark.table("genie_zeroops_mfg_catalog.default.maintenance_manual_parsed").where("parse_ok").collect():
    text = d.full_text
    header = text.split("TASK ", 1)[0]
    engine = (re.search(r"Engine:\s*([^.]+)\.", header) or [None, None])[1]
    # Each troubleshooting task starts with 'TASK nn-nn-nn'
    for part in re.split(r"(?=TASK \d{2}-\d{2}-\d{2})", text)[1:]:
        body = part.strip()
        title = _clean(re.match(r"(TASK .+?)(?=\n\s*Fault Code:)", body, re.S).group(1))
        ata = re.match(r"TASK (\d{2}-\d{2}-\d{2})", body).group(1)
        ac = FIELD("Aircraft Type", body).strip()
        steps = [_clean(s) for s in re.findall(r"Step \d+\.\s*(.+?)(?=\n\s*Step \d+\.|\n\s*Torque Values:|\Z)", body, re.S)]
        rows.append(Row(
            chunk_id=f"{ac}|{ata}",
            manual_name=d.file_name,
            source_path=d.file_path,
            aircraft_type=ac,
            engine=engine,
            ata_chapter=ata,
            task_title=title,
            fault_code=FIELD("Fault Code", body).strip(),
            system=FIELD("System", body).strip(),
            part_name=FIELD("Part Name", body).strip(),
            part_number=FIELD("Part Number", body).strip(),
            related_parts=(FIELD("Related Parts", body) or "").strip(),
            symptoms=(FIELD("Symptoms", body) or "").strip(),
            probable_causes=(FIELD("Probable Causes", body) or "").strip(),
            limits=(FIELD("Limits", body) or "").strip(),
            corrective_steps="\n".join(s.strip() for s in steps),
            torque_values=(FIELD("Torque Values", body) or "").strip(),
            content=body,
            # text that gets embedded: title + key metadata + body
            search_text=f"{title}. Aircraft {ac}. System {FIELD('System', body)}. Part {FIELD('Part Name', body)} "
                        f"{FIELD('Part Number', body)}. Fault {FIELD('Fault Code', body)}.\n{body}",
        ))

chunks = spark.createDataFrame(rows)
(chunks.write.mode("overwrite").option("overwriteSchema", "true")
       .option("delta.enableChangeDataFeed", "true").saveAsTable(CHUNK_TABLE))
spark.sql(f"ALTER TABLE {CHUNK_TABLE} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")
spark.sql(f"COMMENT ON TABLE {CHUNK_TABLE} IS 'One row per troubleshooting task from the synthetic aircraft maintenance manuals. Source for maintenance_manual_index.'")
print(f"{CHUNK_TABLE}: {chunks.count()} chunks from {chunks.select('manual_name').distinct().count()} manuals")
display(spark.table(CHUNK_TABLE).groupBy("fault_code").count().orderBy("fault_code"))

In [0]:
import time
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.vectorsearch import (EndpointType, VectorIndexType, DeltaSyncVectorIndexSpecRequest,
                                                  EmbeddingSourceColumn, PipelineType)

w = WorkspaceClient()
VS_ENDPOINT = "aircraft-manuals-vs"
VS_INDEX = "genie_zeroops_mfg_catalog.default.maintenance_manual_index"
CHUNK_TABLE = "genie_zeroops_mfg_catalog.default.maintenance_manual_chunks"

# 1. Small STANDARD endpoint (create once)
if VS_ENDPOINT not in [e.name for e in w.vector_search_endpoints.list_endpoints()]:
    w.vector_search_endpoints.create_endpoint(name=VS_ENDPOINT, endpoint_type=EndpointType.STANDARD)
while (st := w.vector_search_endpoints.get_endpoint(VS_ENDPOINT).endpoint_status.state.value) != "ONLINE":
    print("endpoint:", st); time.sleep(30)
print("endpoint ONLINE")

# 2. Delta-sync index with Databricks-managed embeddings
existing = [i.name for i in w.vector_search_indexes.list_indexes(endpoint_name=VS_ENDPOINT)]
if VS_INDEX not in existing:
    w.vector_search_indexes.create_index(
        name=VS_INDEX, endpoint_name=VS_ENDPOINT, primary_key="chunk_id",
        index_type=VectorIndexType.DELTA_SYNC,
        delta_sync_index_spec=DeltaSyncVectorIndexSpecRequest(
            source_table=CHUNK_TABLE, pipeline_type=PipelineType.TRIGGERED,
            embedding_source_columns=[EmbeddingSourceColumn(name="search_text", embedding_model_endpoint_name="databricks-gte-large-en")]))
else:
    w.vector_search_indexes.sync_index(index_name=VS_INDEX)

while not (idx := w.vector_search_indexes.get_index(VS_INDEX)).status.ready:
    print("index:", idx.status.message, idx.status.indexed_row_count); time.sleep(30)
print("index ready, rows:", idx.status.indexed_row_count)

In [0]:
res = w.vector_search_indexes.query_index(
    index_name=VS_INDEX,
    columns=["aircraft_type", "task_title", "part_number", "fault_code"],
    query_text="left main gear brake very hot after landing, fuse plug melted",
    query_type="HYBRID", num_results=5,
    filters_json='{"aircraft_type": "B737-800"}')
for r in res.result.data_array:
    print(r)

In [0]:
%sql
-- aircraft-maintenance-app service principal (application id)
GRANT USE CATALOG ON CATALOG genie_zeroops_mfg_catalog TO `a048a692-d417-4d9f-a196-84205b8314b3`;
GRANT USE SCHEMA ON SCHEMA genie_zeroops_mfg_catalog.default TO `a048a692-d417-4d9f-a196-84205b8314b3`;
GRANT SELECT ON TABLE genie_zeroops_mfg_catalog.default.maintenance_manual_chunks TO `a048a692-d417-4d9f-a196-84205b8314b3`;
GRANT SELECT ON TABLE genie_zeroops_mfg_catalog.default.maintenance_manual_index TO `a048a692-d417-4d9f-a196-84205b8314b3`;
GRANT READ VOLUME ON VOLUME genie_zeroops_mfg_catalog.default.maintenance_manuals TO `a048a692-d417-4d9f-a196-84205b8314b3`;